<a href="https://colab.research.google.com/github/iespinozahDM/UBA-DM/blob/main/Copia_de_Clase_02_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clase 2: SQL y Pandas (SQLite)

En esta notebook veremos cómo interactuar con bases de datos relacionales utilizando **SQLite** (un motor de base de datos ligero que no requiere instalación de servidor) y cómo integrar estas consultas con **Pandas**.

## 1. Librerías Necesarias

Python ya trae incluida la librería `sqlite3` para trabajar con archivos `.db`.

In [ ]:
import sqlite3
import pandas as pd

## 2. Crear una Base de Datos y una Tabla

Primero creamos una conexión a un archivo de base de datos (si no existe, se creará).

In [ ]:
conn = sqlite3.connect('clase2_ejemplo.db')
cursor = conn.cursor()

# Crear una tabla
cursor.execute('''
CREATE TABLE IF NOT EXISTS productos (
    id INTEGER PRIMARY KEY,
    nombre TEXT,
    precio REAL,
    categoria TEXT
)
''')

# Insertar algunos datos
datos = [
    (1, 'Notebook', 1500, 'Electrónica'),
    (2, 'Monitor', 300, 'Electrónica'),
    (3, 'Teclado', 50, 'Electrónica'),
    (4, 'Silla Ergonómica', 200, 'Muebles')
]
cursor.executemany('INSERT OR REPLACE INTO productos VALUES (?,?,?,?)', datos)
conn.commit()
print("Datos insertados correctamente.")

Datos insertados correctamente.


## 3. Leer SQL con Pandas

Pandas permite ejecutar una consulta SQL y obtener los resultados directamente en un DataFrame usando `pd.read_sql`.

In [ ]:
query = "SELECT * FROM productos WHERE precio > 100"
df_filtrado = pd.read_sql(query, conn)
df_filtrado

,id,nombre,precio,categoria
0,1,Notebook,1500.0,Electrónica
1,2,Monitor,300.0,Electrónica
2,4,Silla Ergonómica,200.0,Muebles


## 4. Escribir un DataFrame a SQL

También podemos hacer el proceso inverso: llevar un DataFrame a una tabla SQL usando `to_sql`.

In [ ]:
df_ventas = pd.DataFrame({
    'id_producto': [1, 2, 1, 4],
    'cantidad': [10, 5, 2, 8],
    'fecha': ['2026-02-01', '2026-02-02', '2026-02-03', '2026-02-03']
})

df_ventas.to_sql('ventas', conn, if_exists='replace', index=False)
print("DataFrame guardado en la base de datos como la tabla 'ventas'.")

DataFrame guardado en la base de datos como la tabla 'ventas'.


## 5. Consultas Combinadas (Joins)

Podemos usar toda la potencia de SQL directamente desde Pandas.

In [ ]:
query_final = """
SELECT v.fecha, p.nombre, p.precio, v.cantidad, (p.precio * v.cantidad) as total
FROM ventas v
JOIN productos p ON v.id_producto = p.id
"""
df_reporte = pd.read_sql(query_final, conn)
df_reporte

,fecha,nombre,precio,cantidad,total
0,2026-02-01,Notebook,1500.0,10,15000.0
1,2026-02-02,Monitor,300.0,5,1500.0
2,2026-02-03,Notebook,1500.0,2,3000.0
3,2026-02-03,Silla Ergonómica,200.0,8,1600.0


## 6. Explorando la Estructura (Metadata)

A veces recibimos una base de datos sin saber qué tablas contiene. En SQLite, podemos consultar una tabla especial llamada `sqlite_master` para obtener esta información.

Además, para no saturar la memoria, usamos `LIMIT` para traer solo unas pocas filas de prueba.

In [ ]:
# 1. Listar todas las tablas de la base de datos
query_tablas = "SELECT name FROM sqlite_master WHERE type='table';"
tablas = pd.read_sql(query_tablas, conn)
print("Tablas encontradas:")
print(tablas)

# 2. Ver las primeras 3 filas de una tabla específica
query_preview = "SELECT * FROM productos LIMIT 3"
df_preview = pd.read_sql(query_preview, conn)
print("\nVista previa de 'productos':")
print(df_preview)

# 3. Usar Pandas para entender los tipos de datos de la tabla
print("\nInformación de la tabla:")
df_preview.info()

Tablas encontradas:
        name
0  productos
1     ventas

Vista previa de 'productos':
   id    nombre  precio    categoria
0   1  Notebook  1500.0  Electrónica
1   2   Monitor   300.0  Electrónica
2   3   Teclado    50.0  Electrónica

Información de la tabla:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         3 non-null      int64  
 1   nombre     3 non-null      object 
 2   precio     3 non-null      float64
 3   categoria  3 non-null      object 
dtypes: float64(1), int64(1), object(2)
memory usage: 228.0+ bytes


## Trabajo en Clase: Exploración de una Base de Datos Desconocida

**Desafío**: Imagina que recibes un archivo de base de datos de un colega pero no tienes la documentación de qué tablas contiene ni qué datos hay en ellas. Tu objetivo es explorar la estructura y extraer información útil.

**Instrucciones**:
1. **Preparación**: Descargar la base de datos `ecommerce_desconocido.db` de [este link](https://drive.google.com/file/d/1XluiJsmmxjnphUfU1VF46IV-tqrk9ZU8/view?usp=sharing) y subirla al entorno de Colab.
2. **Conexión**: Conectarse a la base de datos creada.
   - *Tip: `sqlite3.connect('nombre.db')`.*
3. **Descubrimiento**: Listar los nombres de todas las tablas existentes.
   - *Tip: Consulta la tabla `sqlite_master`.*
4. **Inspección**: Elegir una tabla y usar `pd.read_sql` con un `LIMIT 5` para entender las columnas.
5. **Análisis**: Cargar la tabla completa en un DataFrame y utilizar el método `.info()` o `.describe()`.
6. **Extracción**: Realizar una consulta SQL que filtre los datos por alguna condición.
   - *Tip: Prueba un `SELECT * FROM tabla WHERE condicion`.*
7. **Avanzado**: Construir un dataframe a partir de un join complejo, por ejemplo, cantidad de productos y nombre de cliente por pedido.
   - *Tip: La query de `read_sql` puede contener joins y cualquier sql compatible con SQLite. Se recomienda probar consultas avanzadas.*

**Resultado esperado**: El código debe mostrar la lista de tablas encontradas y un breve resumen estadístico de la tabla principal analizada.

In [ ]:
# Librerias
import pandas as pd
import sqlite3

import sklearn as sk
from sklearn import model_selection
from sklearn import ensemble
from sklearn import metrics

In [ ]:
#Montar y cargar archivo
from google.colab import drive
drive.mount("/content/drive")
# TODO: Cambiar para que apunte al directorio correcto
DIR = "/content/drive/MyDrive/DM/"
engine = sqlite3.connect(f"{DIR}/ecommerce_desconocido.db")

Mounted at /content/drive


In [ ]:
# Indagar en las tablas de la DB
query_tablas = "SELECT name FROM sqlite_master WHERE type='table';"
tablas = pd.read_sql(query_tablas, engine)
print("Tablas encontradas:")
print(tablas)

Tablas encontradas:
            name
0       clientes
1      productos
2        pedidos
3  pedidos_items


In [ ]:
# Ver las primeras 5 filas de una tabla específica
query_preview = "SELECT * FROM productos LIMIT 6"
df_preview = pd.read_sql(query_preview, engine)
print("\nVista previa de 'productos':")
print(df_preview)


Vista previa de 'productos':
   id      nombre  precio categoria
0   1  Producto 1   99.09  Juguetes
1   2  Producto 2  312.57     Hogar
2   3  Producto 3  267.13  Deportes
3   4  Producto 4   21.30      Ropa
4   5  Producto 5  309.81     Hogar
5   6  Producto 6   32.87  Deportes


In [ ]:
df_prod=pd.read_sql("SELECT * FROM productos", engine)
df_prod

,id,nombre,precio,categoria
0,1,Producto 1,99.09,Juguetes
1,2,Producto 2,312.57,Hogar
2,3,Producto 3,267.13,Deportes
3,4,Producto 4,21.30,Ropa
4,5,Producto 5,309.81,Hogar
5,6,Producto 6,32.87,Deportes
6,7,Producto 7,189.52,Ropa
7,8,Producto 8,197.41,Deportes
8,9,Producto 9,300.28,Ropa
9,10,Producto 10,431.37,Juguetes


In [ ]:
#Cargar otras tablas
#df_ped=pd.read_sql("SELECT * FROM pedidos", engine)
#df_ped
#df_pitem=pd.read_sql("SELECT * FROM pedidos_items", engine)
#df_pitem
df_cli=pd.read_sql("SELECT * FROM clientes", engine)
df_cli

,id,nombre,ciudad,email
0,1,Lucía,Mendoza,lucía1@ejemplo.com
1,2,Sofía,Córdoba,sofía2@ejemplo.com
2,3,Diego,La Plata,diego3@ejemplo.com
3,4,Sofía,Rosario,sofía4@ejemplo.com
4,5,Marta,Córdoba,marta5@ejemplo.com
5,6,Marta,La Plata,marta6@ejemplo.com
6,7,Juan,Córdoba,juan7@ejemplo.com
7,8,Pedro,La Plata,pedro8@ejemplo.com
8,9,Luis,Mendoza,luis9@ejemplo.com
9,10,Pedro,Rosario,pedro10@ejemplo.com


In [ ]:
#Queries
cons = "SELECT * FROM productos WHERE precio<200"
df_cons = pd.read_sql(cons, engine)
print("\nVista previa de 'productos':")
print(df_cons)



Vista previa de 'productos':
   id       nombre  precio categoria
0   1   Producto 1   99.09  Juguetes
1   4   Producto 4   21.30      Ropa
2   6   Producto 6   32.87  Deportes
3   7   Producto 7  189.52      Ropa
4   8   Producto 8  197.41  Deportes
5  14  Producto 14  159.26  Juguetes
6  15  Producto 15  123.14  Deportes
7  19  Producto 19   99.30  Deportes
8  20  Producto 20  162.74     Hogar


In [ ]:
query_final = """
SELECT i.id, i.nombre, p.total
FROM clientes i
JOIN pedidos p ON p.id_cliente = i.id
"""
df_reporte = pd.read_sql(query_final, engine)
df_reporte




,id,nombre,total
0,4,Sofía,267.13
1,9,Luis,998.85
2,7,Juan,926.50
3,7,Juan,1794.24
4,6,Marta,432.95
5,3,Diego,1818.05
6,1,Lucía,692.22
7,1,Lucía,1797.08
8,8,Pedro,282.40
9,3,Diego,2439.61


In [ ]:
# Cerramos la conexión al terminar
engine.close()